## Phase 2 ##
- Missile Data lookup
- Missile patterning 
- Geo Spaces and open sourced information 

In [11]:
# phase_2Jun3v3

# Scrape through missle databases to localize the data and find the right ones to use and make sure it works for use case 
# Need to also setup spacialized spaces

In [12]:
import numpy as np
import plotly.graph_objects as go

# --- 1. Physics Engine ---
def missile_dynamics(state, t, accel_cmd=np.zeros(3)):
    vel = state[3:6]
    gravity_accel = np.array([0.0, 0.0, -9.81])
    
    # Zeroing drag for the initial ballistic vacuum test at scale
    drag_accel = np.array([0.0, 0.0, 0.0]) 
    
    accel = gravity_accel + drag_accel + accel_cmd
    return np.concatenate((vel, accel))

def rk4_step(state, t, dt, derivatives_fn, accel_cmd=np.zeros(3)):
    k1 = derivatives_fn(state, t, accel_cmd)
    k2 = derivatives_fn(state + 0.5 * dt * k1, t + 0.5 * dt, accel_cmd)
    k3 = derivatives_fn(state + 0.5 * dt * k2, t + 0.5 * dt, accel_cmd)
    k4 = derivatives_fn(state + dt * k3, t + dt, accel_cmd)
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

In [13]:
# --- 2. Geo Space Definitions (in meters) ---
# Space 1: 1,600km x 1,600km (Origin)
s1_min, s1_max = -800000, 800000 

# Space 2: 500km x 500km (Located 4,000km downrange on the X-axis)
s2_center_x = 4000000 
s2_half_side = 250000
s2_x_min, s2_x_max = s2_center_x - s2_half_side, s2_center_x + s2_half_side
s2_y_min, s2_y_max = -s2_half_side, s2_half_side

# --- 3. Initial Launch State ---
# Launching from the center of Space 1, aiming at the center of Space 2
# State: [x, y, z, vx, vy, vz]
# vx = 3800 m/s, vz = 5200 m/s yields a flight time of ~1060s and range of ~4000km
missile_state = np.array([0.0, 0.0, 0.0, 3800.0, 0.0, 5200.0]) 

dt = 1.0  # Increased time step to 1 second because the flight is long
total_time = 1100  
steps = int(total_time / dt)

missile_path = np.zeros((steps, 3))

# --- 4. Simulation Loop ---
for i in range(steps):
    missile_path[i] = missile_state[0:3]
    t = i * dt
    
    missile_state = rk4_step(missile_state, t, dt, missile_dynamics)
    
    # Stop recording if it hits the ground
    if missile_state[2] < 0 and i > 10: 
        missile_path = missile_path[:i] # Truncate array to actual flight time
        steps = i
        break

print(f"--- Surface-to-Surface Launch ---")
print(f"Flight Time: {steps * dt} seconds")
print(f"Impact Coordinates: X: {missile_path[-1,0]:.0f}m, Y: {missile_path[-1,1]:.0f}m")

--- Surface-to-Surface Launch ---
Flight Time: 1060.0 seconds
Impact Coordinates: X: 4024200m, Y: 0m


In [14]:
# --- 5. 3D Plotting at Scale ---
fig = go.Figure()

# Draw Space 1 (Green)
fig.add_trace(go.Mesh3d(
    x=[s1_min, s1_max, s1_max, s1_min],
    y=[s1_min, s1_min, s1_max, s1_max],
    z=[0, 0, 0, 0],
    color='rgba(0, 255, 0, 0.2)',
    name='Space 1 (Launch)'
))

# Draw Space 2 (Red)
fig.add_trace(go.Mesh3d(
    x=[s2_x_min, s2_x_max, s2_x_max, s2_x_min],
    y=[s2_y_min, s2_y_min, s2_y_max, s2_y_max],
    z=[0, 0, 0, 0],
    color='rgba(255, 0, 0, 0.2)',
    name='Space 2 (Target)'
))

# Plot Ballistic Trajectory
fig.add_trace(go.Scatter3d(
    x=missile_path[:,0], y=missile_path[:,1], z=missile_path[:,2],
    mode='lines', line=dict(color='orange', width=4), name='Ballistic Arc'
))

# Animated Missile Marker
fig.add_trace(go.Scatter3d(
    x=[missile_path[0,0]], y=[missile_path[0,1]], z=[missile_path[0,2]], 
    mode='markers', marker=dict(size=6, color='red'), name='Missile'
))

# Animation Frames
frame_skip = 20 # Skipping frames to keep animation fast over 1000+ seconds
frames = []
for k in range(0, steps, frame_skip):
    frames.append(go.Frame(
        data=[go.Scatter3d(x=[missile_path[k,0]], y=[missile_path[k,1]], z=[missile_path[k,2]])],
        traces=[3], # Update only the missile marker
        name=f'frame_{k}'
    ))
fig.frames = frames

fig.update_layout(
    title='Phase 2: Surface-to-Surface Macro Scale Launch',
    scene=dict(
        xaxis_title='X (meters)',
        yaxis_title='Y (meters)',
        zaxis_title='Altitude (meters)',
        aspectmode='data' # Forces axes to scale proportionally to real distance
    ),
    updatemenus=[dict(
        type="buttons",
        buttons=[
            dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=20, redraw=True), fromcurrent=True)]),
            dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])
        ]
    )]
)

fig.show()

In [15]:
def compute_cpa(pos1, vel1, pos2, vel2):
    dp = pos1 - pos2
    dv = vel1 - vel2
    dv_sq = max(np.dot(dv, dv), 1e-8) 
    t_cpa = -np.dot(dp, dv) / dv_sq
    
    if t_cpa < 0:
        return 0.0, np.linalg.norm(dp)
        
    pos1_cpa = pos1 + vel1 * t_cpa
    pos2_cpa = pos2 + vel2 * t_cpa
    miss_dist = np.linalg.norm(pos1_cpa - pos2_cpa)
    return t_cpa, miss_dist

def proportional_navigation_macro(interceptor_state, target_state, N=5.0):
    r_vec = target_state[0:3] - interceptor_state[0:3]
    v_rel = target_state[3:6] - interceptor_state[3:6]
    
    r = max(np.linalg.norm(r_vec), 1e-6)
    r_hat = r_vec / r
    omega = np.cross(r_vec, v_rel) / (r ** 2)
    
    v_mag = np.linalg.norm(interceptor_state[3:6])
    accel_cmd = N * v_mag * np.cross(omega, r_hat)
    
    # Scaled up for Sprint/Exoatmospheric interceptors (50 Gs)
    max_accel = 50.0 * 9.81
    if np.linalg.norm(accel_cmd) > max_accel:
        accel_cmd = (accel_cmd / np.linalg.norm(accel_cmd)) * max_accel
        
    return accel_cmd

# --- 2. Physics Engine ---
def missile_dynamics(state, t, accel_cmd=np.zeros(3)):
    vel = state[3:6]
    gravity_accel = np.array([0.0, 0.0, -9.81])
    accel = gravity_accel + accel_cmd
    return np.concatenate((vel, accel))

def rk4_step(state, t, dt, derivatives_fn, accel_cmd=np.zeros(3)):
    k1 = derivatives_fn(state, t, accel_cmd)
    k2 = derivatives_fn(state + 0.5 * dt * k1, t + 0.5 * dt, accel_cmd)
    k3 = derivatives_fn(state + 0.5 * dt * k2, t + 0.5 * dt, accel_cmd)
    k4 = derivatives_fn(state + dt * k3, t + dt, accel_cmd)
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

In [16]:
# --- 3. Initial States ---
# ICBM: Launching from Origin (Space 1) towards Space 2
target_state = np.array([0.0, 0.0, 0.0, 3800.0, 0.0, 5200.0]) 

# Interceptor: Forward-deployed 500km downrange, 50km offset. 
# Launching aggressively upwards and slightly backwards towards the target.
interceptor_state = np.array([500000.0, 50000.0, 0.0, -1500.0, -200.0, 6500.0])

dt = 0.5  # 0.5 second resolution for macro scale
total_time = 300  # We only need ~300 seconds for a Boost-Phase Intercept
steps = int(total_time / dt)

target_path = np.zeros((steps, 3))
interceptor_path = np.zeros((steps, 3))
distances = np.zeros(steps)

# --- 4. Simulation Loop ---
hit_registered = False
actual_steps = steps

for i in range(steps):
    target_path[i] = target_state[0:3]
    interceptor_path[i] = interceptor_state[0:3]
    distances[i] = np.linalg.norm(target_path[i] - interceptor_path[i])
    
    t = i * dt
    
    # Active Tracking
    interceptor_accel = proportional_navigation_macro(interceptor_state, target_state, N=5.0)
    
    # Step Physics
    target_state = rk4_step(target_state, t, dt, missile_dynamics)
    interceptor_state = rk4_step(interceptor_state, t, dt, missile_dynamics, accel_cmd=interceptor_accel)
    
    # Break early if interception occurs (Distance < 500m at these massive speeds)
    if distances[i] < 500.0 and i > 5:
        actual_steps = i + 1
        hit_registered = True
        break

# Truncate arrays to actual flight time
target_path = target_path[:actual_steps]
interceptor_path = interceptor_path[:actual_steps]
distances = distances[:actual_steps]

# --- 5. CPA Analysis ---
min_dist_idx = np.argmin(distances)
time_of_interception = min_dist_idx * dt
interception_coords = target_path[min_dist_idx]

# Sub-timestep precision
t_cpa, precise_miss_dist = compute_cpa(
    target_path[-1], target_state[3:6], 
    interceptor_path[-1], interceptor_state[3:6]
)

print(f"--- Boost-Phase Intercept Results ---")
print(f"Time of Intercept:    {time_of_interception:.1f} seconds")
print(f"Altitude of Kill:     {interception_coords[2]/1000:.1f} kilometers")
print(f"Downrange Distance:   {interception_coords[0]/1000:.1f} kilometers (Inside Space 1)")
print(f"Geometric CPA Miss:   {precise_miss_dist:.2f} meters")

if hit_registered or precise_miss_dist <= 500.0:
    print(f"STATUS: KINETIC KILL CONFIRMED")
else:
    print(f"STATUS: MISS")

--- Boost-Phase Intercept Results ---
Time of Intercept:    95.5 seconds
Altitude of Kill:     451.9 kilometers
Downrange Distance:   362.9 kilometers (Inside Space 1)
Geometric CPA Miss:   56.17 meters
STATUS: KINETIC KILL CONFIRMED


In [17]:
# --- 6. 3D Plotting ---
s1_min, s1_max = -800000, 800000 
s2_center_x = 4000000 
s2_half_side = 250000

fig = go.Figure()

# Space 1 (Launch Area)
fig.add_trace(go.Mesh3d(x=[s1_min, s1_max, s1_max, s1_min], y=[s1_min, s1_min, s1_max, s1_max], z=[0, 0, 0, 0], color='rgba(0, 255, 0, 0.1)', name='Space 1'))
# Space 2 (Target Area)
fig.add_trace(go.Mesh3d(x=[s2_center_x-s2_half_side, s2_center_x+s2_half_side, s2_center_x+s2_half_side, s2_center_x-s2_half_side], y=[-s2_half_side, -s2_half_side, s2_half_side, s2_half_side], z=[0, 0, 0, 0], color='rgba(255, 0, 0, 0.1)', name='Space 2'))

# Trajectories
fig.add_trace(go.Scatter3d(x=target_path[:,0], y=target_path[:,1], z=target_path[:,2], mode='lines', line=dict(color='rgba(255,0,0,0.5)', width=4), name='ICBM Track'))
fig.add_trace(go.Scatter3d(x=interceptor_path[:,0], y=interceptor_path[:,1], z=interceptor_path[:,2], mode='lines', line=dict(color='rgba(0,0,255,0.5)', width=4), name='Interceptor Track'))

# Detonation Point
if hit_registered or precise_miss_dist <= 500.0:
    fig.add_trace(go.Scatter3d(x=[interception_coords[0]], y=[interception_coords[1]], z=[interception_coords[2]], mode='markers', marker=dict(size=12, color='yellow', symbol='x'), name='Kinetic Kill Point'))

# Animated Markers
base_traces = len(fig.data)
target_idx = base_traces
interceptor_idx = base_traces + 1

fig.add_trace(go.Scatter3d(x=[target_path[0,0]], y=[target_path[0,1]], z=[target_path[0,2]], mode='markers', marker=dict(size=6, color='red'), name='ICBM'))
fig.add_trace(go.Scatter3d(x=[interceptor_path[0,0]], y=[interceptor_path[0,1]], z=[interceptor_path[0,2]], mode='markers', marker=dict(size=6, color='blue'), name='Interceptor'))

frame_skip = max(1, int(actual_steps / 60)) # Dynamic frame skip for smooth animation
frames = []
for k in range(0, actual_steps, frame_skip):
    frames.append(go.Frame(
        data=[
            go.Scatter3d(x=[target_path[k,0]], y=[target_path[k,1]], z=[target_path[k,2]]),
            go.Scatter3d(x=[interceptor_path[k,0]], y=[interceptor_path[k,1]], z=[interceptor_path[k,2]])
        ],
        traces=[target_idx, interceptor_idx],
        name=f'frame_{k}'
    ))
# Add the final frame to ensure the hit is shown
frames.append(go.Frame(data=[go.Scatter3d(x=[target_path[-1,0]], y=[target_path[-1,1]], z=[target_path[-1,2]]), go.Scatter3d(x=[interceptor_path[-1,0]], y=[interceptor_path[-1,1]], z=[interceptor_path[-1,2]])], traces=[target_idx, interceptor_idx], name='frame_final'))
fig.frames = frames

fig.update_layout(
    title='Phase 2: Boost-Phase Intercept (Forward Deployed)',
    scene=dict(
        xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Altitude (m)',
        aspectmode='data'
    ),
    updatemenus=[dict(type="buttons", buttons=[dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=20, redraw=True), fromcurrent=True)]), dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])]
)

fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# =====================================================================
# 1. MATHEMATICAL & GEOMETRIC UTILITIES
# =====================================================================

def compute_cpa(pos1, vel1, pos2, vel2):
    """
    Computes the closest point of approach (CPA) time and precise miss distance
    between two moving vectors using sub-timestep linear interpolation.
    """
    dp = pos1 - pos2
    dv = vel1 - vel2
    dv_sq = max(np.dot(dv, dv), 1e-8) 
    t_cpa = -np.dot(dp, dv) / dv_sq
    
    if t_cpa < 0:
        return 0.0, np.linalg.norm(dp)
        
    pos1_cpa = pos1 + vel1 * t_cpa
    pos2_cpa = pos2 + vel2 * t_cpa
    miss_dist = np.linalg.norm(pos1_cpa - pos2_cpa)
    return t_cpa, miss_dist

def proportional_navigation_macro(interceptor_state, target_state, N=4.5):
    """
    Implements Proportional Navigation (ProNav) guidance.
    Calculates lateral acceleration commands based on line-of-sight rate.
    """
    r_vec = target_state[0:3] - interceptor_state[0:3]
    v_rel = target_state[3:6] - interceptor_state[3:6]
    
    r = max(np.linalg.norm(r_vec), 1e-6)
    r_hat = r_vec / r
    omega = np.cross(r_vec, v_rel) / (r ** 2)
    
    v_mag = np.linalg.norm(interceptor_state[3:6])
    accel_cmd = N * v_mag * np.cross(omega, r_hat)
    
    # Defensive high-G maneuver cap for high-velocity interception
    max_accel = 40.0 * 9.81
    if np.linalg.norm(accel_cmd) > max_accel:
        accel_cmd = (accel_cmd / np.linalg.norm(accel_cmd)) * max_accel
        
    return accel_cmd

# =====================================================================
# 2. PHYSICS ENGINE (DYNAMIC DIFFERENTIAL EQUATIONS & RK4)
# =====================================================================

def missile_dynamics(state, t, accel_cmd=np.zeros(3)):
    """
    Returns the derivative of the state vector [vx, vy, vz, ax, ay, az].
    Applies constant gravitational acceleration.
    """
    vel = state[3:6]
    gravity_accel = np.array([0.0, 0.0, -9.81])
    accel = gravity_accel + accel_cmd
    return np.concatenate((vel, accel))

def rk4_step(state, t, dt, derivatives_fn, accel_cmd=np.zeros(3)):
    """
    4th-Order Runge-Kutta integration step for high-precision projection.
    """
    k1 = derivatives_fn(state, t, accel_cmd)
    k2 = derivatives_fn(state + 0.5 * dt * k1, t + 0.5 * dt, accel_cmd)
    k3 = derivatives_fn(state + 0.5 * dt * k2, t + 0.5 * dt, accel_cmd)
    k4 = derivatives_fn(state + dt * k3, t + dt, accel_cmd)
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

# =====================================================================
# 3. ENGAGEMENT SIMULATION CONFIGURATION
# =====================================================================

target_state = np.array([0.0, 0.0, 0.0, 3800.0, 0.0, 5200.0]) 

silo_x = 3750000.0
silo_y = 0.0
silo_z = 0.0
interceptor_state = np.array([silo_x, silo_y, silo_z, 0.0, 0.0, 0.0])

dt = 0.5  
total_time = 1100  
steps = int(total_time / dt)

target_path = np.zeros((steps, 3))
interceptor_path = np.zeros((steps, 3))
distances = np.zeros(steps)

interceptor_launched = False
hit_registered = False
actual_steps = steps

# FIX 1: Expanded Radar Horizon (1,500 kilometers)
launch_threshold_range = 1500000.0 

# =====================================================================
# 4. SIMULATION EXECUTION LOOP
# =====================================================================

for i in range(steps):
    target_path[i] = target_state[0:3]
    t = i * dt
    
    range_to_target = np.linalg.norm(target_state[0:3] - np.array([silo_x, silo_y, silo_z]))
    
    if not interceptor_launched and range_to_target <= launch_threshold_range:
        interceptor_launched = True
        # FIX 2: Massive boost to initial velocity (Mach 20+) to reach the exosphere
        interceptor_state[3:6] = np.array([-3500.0, 0.0, 6500.0])
        print(f"[RADAR] Early Warning Track Acquired at t = {t:.1f}s. Interceptor Launched!")

    if interceptor_launched:
        interceptor_path[i] = interceptor_state[0:3]
        distances[i] = np.linalg.norm(target_path[i] - interceptor_path[i])
        
        interceptor_accel = proportional_navigation_macro(interceptor_state, target_state, N=4.5)
        interceptor_state = rk4_step(interceptor_state, t, dt, missile_dynamics, accel_cmd=interceptor_accel)
        
        if distances[i] < 500.0:
            actual_steps = i + 1
            hit_registered = True
            break
    else:
        interceptor_path[i] = np.array([silo_x, silo_y, silo_z])
        distances[i] = range_to_target

    target_state = rk4_step(target_state, t, dt, missile_dynamics)
    
    if target_state[2] < 0:
        actual_steps = i + 1
        break

target_path = target_path[:actual_steps]
interceptor_path = interceptor_path[:actual_steps]
distances = distances[:actual_steps]

# =====================================================================
# 5. DATA POST-PROCESSING & ENGAGEMENT ANALYSIS
# =====================================================================

min_dist_idx = np.argmin(distances)
time_of_interception = min_dist_idx * dt
interception_coords = target_path[min_dist_idx]

t_cpa, precise_miss_dist = compute_cpa(
    target_path[-1], target_state[3:6], 
    interceptor_path[-1], interceptor_state[3:6]
)

print(f"\n--- Terminal Intercept Engagement Summary ---")
print(f"Interception Time:     {time_of_interception:.1f} seconds")
print(f"Interception Altitude: {interception_coords[2]/1000:.1f} km (Meso / Exo-atmospheric)")
print(f"Downrange Distance:    {interception_coords[0]/1000:.1f} km (Edge of Space 2)")
print(f"Geometric CPA Miss:    {precise_miss_dist:.2f} meters")

if hit_registered or precise_miss_dist <= 20.0:
    print(f"STATUS: SUCCESSFUL AREA INTERCEPTION")
else:
    print(f"STATUS: LEAK - THREAT COMPROMISED PROTECTED ZONE")

# =====================================================================
# 6. MACRO-SCALE 3D GEOSPATIAL VISUALIZATION
# =====================================================================

s1_min, s1_max = -800000, 800000 
s2_center_x = 4000000 
s2_half_side = 250000
s2_x_min, s2_x_max = s2_center_x - s2_half_side, s2_center_x + s2_half_side

fig = go.Figure()

# Ground Space 1 (Launch Zone Mesh)
fig.add_trace(go.Mesh3d(x=[s1_min, s1_max, s1_max, s1_min], y=[s1_min, s1_min, s1_max, s1_max], z=[0, 0, 0, 0], color='rgba(0, 255, 0, 0.1)', name='Space 1'))
# Ground Space 2 (Defended Zone Mesh)
fig.add_trace(go.Mesh3d(x=[s2_x_min, s2_x_max, s2_x_max, s2_x_min], y=[-s2_half_side, -s2_half_side, s2_half_side, s2_half_side], z=[0, 0, 0, 0], color='rgba(255, 0, 0, 0.1)', name='Space 2'))

# Continuous Flight Tracks
fig.add_trace(go.Scatter3d(x=target_path[:,0], y=target_path[:,1], z=target_path[:,2], mode='lines', line=dict(color='rgba(255,0,0,0.5)', width=3), name='Threat Vector'))
fig.add_trace(go.Scatter3d(x=interceptor_path[:,0], y=interceptor_path[:,1], z=interceptor_path[:,2], mode='lines', line=dict(color='rgba(0,0,255,0.6)', width=3), name='Defensive Vector'))

# Kinetic Blast Point Marker
if hit_registered:
    fig.add_trace(go.Scatter3d(x=[interception_coords[0]], y=[interception_coords[1]], z=[interception_coords[2]], mode='markers', marker=dict(size=12, color='orange', symbol='diamond'), name='Terminal Intercept Point'))

# Dynamic Markers for Animation
base_traces = len(fig.data)
target_idx = base_traces
interceptor_idx = base_traces + 1

fig.add_trace(go.Scatter3d(x=[target_path[0,0]], y=[target_path[0,1]], z=[target_path[0,2]], mode='markers', marker=dict(size=6, color='red'), name='Target'))
fig.add_trace(go.Scatter3d(x=[interceptor_path[0,0]], y=[interceptor_path[0,1]], z=[interceptor_path[0,2]], mode='markers', marker=dict(size=6, color='blue'), name='Interceptor'))

# Build Animation Frames
frame_skip = max(1, int(actual_steps / 80))
frames = []
for k in range(0, actual_steps, frame_skip):
    frames.append(go.Frame(
        data=[
            go.Scatter3d(x=[target_path[k,0]], y=[target_path[k,1]], z=[target_path[k,2]]),
            go.Scatter3d(x=[interceptor_path[k,0]], y=[interceptor_path[k,1]], z=[interceptor_path[k,2]])
        ],
        traces=[target_idx, interceptor_idx],
        name=f'frame_{k}'
    ))
fig.frames = frames

# Chart Formatting Layout Config
fig.update_layout(
    title='Phase 2: Terminal Area Defense Intercept (Edge of Space 2)',
    scene=dict(
        xaxis_title='X (mete s)', yaxis_title='Y (meters)', zaxis_title='Altitude (meters)',
        aspectmode='data'
    ),
    updatemenus=[dict(type="buttons", buttons=[dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=15, redraw=True), fromcurrent=True)]), dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])]
)

fig.show()

[RADAR] Early Warning Track Acquired at t = 738.5s. Interceptor Launched!

--- Terminal Intercept Engagement Summary ---
Interception Time:     872.0 seconds
Interception Altitude: 804.7 km (Meso / Exo-atmospheric)
Downrange Distance:    3313.6 km (Edge of Space 2)
Geometric CPA Miss:    254.61 meters
STATUS: SUCCESSFUL AREA INTERCEPTION
